# 📊 US AR Aging 原始数据

从 Excel 透视表提取的原始应收账款账龄数据。

| 列名 | 说明 |
|------|------|
| Customer | 客户名称 |
| Currency | 币种 |
| 0 | 当期（Current） |
| 30 | 30天账龄 |
| 90 | 90天账龄 |
| 180 | 180天账龄 |
| 360 | 360天账龄 |
| 1+ Year | 1年以上 |
| Credit Note | 贷项通知 |
| Grand Total | 合计 |

In [4]:
import pandas as pd
import numpy as np

# 文件路径
file_path = "/Users/jamesl/Library/CloudStorage/GoogleDrive-nxtagentimes@gmail.com/我的云端硬盘/6. CMI/数据集/US AR Aging_20260624.xlsx"

# 读取第一个 sheet，跳过前3行标题，第4行作为列名
df_raw = pd.read_excel(file_path, sheet_name=0, header=None)

# 找到实际的 header 行（包含 "Customer" 的行）
header_row = None
for i, row in df_raw.iterrows():
    if any(str(v).strip() == "Customer" for v in row.values if pd.notna(v)):
        header_row = i
        break

print(f"📋 Header 行号: {header_row}")

# 用找到的 header 行读取数据
df = pd.read_excel(file_path, sheet_name=0, header=header_row)

# 列名转为字符串（数字列如 0, 30 等会被 pandas 自动转为数字）
df.columns = [str(c).strip() for c in df.columns]
print(f"📋 列名: {df.columns.tolist()}")
print(f"📊 原始数据: {len(df)} 行 × {len(df.columns)} 列")

📋 Header 行号: 3
📋 列名: ['Customer', 'Currency', '0', '30', '90', '180', '360', '1+ Year', 'Credit Note', 'Grand Total']
📊 原始数据: 271 行 × 10 列


## 加载与清洗

- 跳过透视表标题行，自动识别 header
- 去除空行和 Grand Total 汇总行
- 金额列转为数值类型

In [6]:
# 金额列
amount_cols = ["0", "30", "90", "180", "360", "1+ Year", "Credit Note", "Grand Total"]
amount_cols = [c for c in amount_cols if c in df.columns]

# 去除空行和 Grand Total 汇总行
df = df[df["Customer"].notna() & (df["Customer"] != "Grand Total")].copy()

# 金额列转数值，NaN 填 0
for col in amount_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

df = df.reset_index(drop=True)
print(f"✅ 共 {len(df)} 个客户")
df

✅ 共 265 个客户


,Customer,Currency,0,30,90,180,360,1+ Year,Credit Note,Grand Total
0,1304748 B.C LTD,USD,0.00,0.00,0.00,0.00,0.00,59741.94,0.00,59741.94
1,8x8 inc,USD,45210.17,19202.83,0.00,0.00,0.00,0.00,0.00,64413.00
2,ACROBAT TELECOM INC.,USD,1.14,0.58,2.48,5.22,0.27,0.00,0.00,9.69
3,ACT international Telecomm Lim,USD,0.00,0.00,0.00,0.00,0.00,0.00,-354.84,-354.84
4,Activ Financial Systems Inc,USD,5300.00,0.00,0.00,0.00,0.00,0.00,0.00,5300.00
...,...,...,...,...,...,...,...,...,...,...
260,江苏金珹智能储存设备有限公司,CNY,0.00,0.00,0.00,0.00,1027110.00,0.00,0.00,1027110.00
261,科赋锐（北京）信息科技有限公司,USD,2900.35,5987.82,0.00,0.00,0.00,0.00,0.00,8888.17
262,雲鐳 (香港)有限公司,USD,40400.00,51235.41,0.00,0.00,0.00,0.00,0.00,91635.41
263,露露樂蒙股份有限公司,USD,250.71,0.00,0.00,0.00,0.00,0.00,0.00,250.71


## 导入数据库

- 从文件名提取导入日期（如 `US AR Aging_20260624.xlsx` → `2026-06-24`）
- 同一天重复导入时，先删除旧数据再写入

In [12]:
# 导入数据库 — 调用 import_aging.py
import subprocess, sys
subprocess.run([sys.executable, "import_aging.py", file_path], check=True)

📂 Excel 文件: /Users/jamesl/Library/CloudStorage/GoogleDrive-nxtagentimes@gmail.com/我的云端硬盘/6. CMI/数据集/US AR Aging_20260624.xlsx
📅 导入日期: 2026-06-24
📊 读取 265 条记录
🗑️  删除旧记录: 265 条

✅ 导入完成！
────────────────────────────────────────
📅 导入日期:  2026-06-24
📋 记录条数:  265
────────────────────────────────────────
  💰 0                 21,664,896.79
  💰 30                 7,985,609.00
  💰 90                14,553,868.06
  💰 180                5,161,443.16
  💰 360                2,005,708.38
  💰 1+ Year           21,977,988.42
  💰 Grand Total       71,925,945.46
────────────────────────────────────────


CompletedProcess(args=['/Users/jamesl/PrjCenter/pesg-starter/learning/python/.venv/bin/python', 'import_aging.py', '/Users/jamesl/Library/CloudStorage/GoogleDrive-nxtagentimes@gmail.com/我的云端硬盘/6. CMI/数据集/US AR Aging_20260624.xlsx'], returncode=0)

## 新产生 Aging 分析

- 对比最新一周与上一周数据，找出 **新产生 aging**（当期金额从无到有）的客户
- 按当期(0)金额从大到小排列
- 仅一期数据时，显示当期所有有金额的客户

In [7]:
import sqlite3

db_path = "/Users/jamesl/Library/CloudStorage/GoogleDrive-nxtagentimes@gmail.com/我的云端硬盘/6. CMI/数据集/cmidata.db"

In [ ]:
import sqlite3
import pandas as pd

db_path = "/Users/jamesl/Library/CloudStorage/GoogleDrive-nxtagentimes@gmail.com/我的云端硬盘/6. CMI/数据集/cmidata.db"
conn = sqlite3.connect(db_path)

# 获取最新日期
latest_date = pd.read_sql_query(
    "SELECT MAX(import_date) AS d FROM ar_aging", conn
)["d"].iloc[0]

# 查询当期有金额的客户
df_export = pd.read_sql_query("""
    SELECT customer, aging_0
    FROM ar_aging
    WHERE import_date = ? AND aging_0 > 0
    ORDER BY aging_0 DESC
""", conn, params=[latest_date])
conn.close()

csv_path = f"ar_aging_{latest_date.replace('-', '')}.csv"
df_export.assign(aging_0=lambda x: x["aging_0"].round(2)).rename(
    columns={"customer": "Customer", "aging_0": "0"}
).to_csv(csv_path, index=False)
print(f"✅ 已导出: {csv_path}（{len(df_export)} 条）")

NameError: name 'show' is not defined